# Train **My Own AI Model** on Google Colab

Trains the from-scratch GPT (`llm-from-scratch/`) to completion on Colab's free compute — which runs a cell uninterrupted, unlike the ephemeral dev container. Produces a `web/model.json` you download and hand back to deploy to the live app at `/llm/`.

**How to use:** Runtime → Run all (or run each cell top to bottom). Training takes ~15–40 min depending on size. The last cell downloads `model.json`.

⚠️ This is a tiny model: it learns the *style* of the training text (Wikipedia → encyclopedic voice), not real facts. Names/dates it produces are invented.

## 1. Get the code + dependencies

In [ ]:
!git clone --depth 1 https://github.com/Refayethossain28/BallrzAPP.git
%cd BallrzAPP/llm-from-scratch
!pip -q install numpy

## 2. Pick a corpus

Default is **WikiText-2** (~10 MB of real Wikipedia article text). Swap `CORPUS_URL` for any plain-text UTF-8 file to change the model's voice (e.g. a Project Gutenberg book). The cleanup strips WikiText's `<unk>` / `@,@` markup so output reads naturally.

In [ ]:
CORPUS_URL = "https://raw.githubusercontent.com/pytorch/examples/main/word_language_model/data/wikitext-2/train.txt"
CLEAN_WIKITEXT = True   # set False for non-WikiText corpora

import urllib.request, re, os
os.makedirs('data', exist_ok=True)
urllib.request.urlretrieve(CORPUS_URL, 'data/corpus.txt')
text = open('data/corpus.txt', encoding='utf-8', errors='ignore').read()
if CLEAN_WIKITEXT:
    text = text.replace('@,@', '').replace(' @.@ ', '.').replace(' @-@ ', '-').replace('<unk>', '')
    text = re.sub(r' ([,.;:!?)])', r'\1', text).replace('( ', '(')
    open('data/corpus.txt', 'w', encoding='utf-8').write(text)
print(f'corpus: {len(text):,} chars')

## 3. Train

Defaults train a **6-layer / 256-dim (~5M param)** model — ~3× the currently-deployed one. Bigger `--n_layer` / `--n_embd` / `--steps` = better text but a larger `model.json` download and slower in-browser generation. This is resumable: if the Colab runtime drops, just run this cell again and it continues from the last checkpoint.

In [ ]:
!python train.py --data data/corpus.txt --tokenizer bpe --vocab_size 1024 \
    --n_layer 6 --n_head 8 --n_embd 256 --block_size 128 --batch_size 16 \
    --steps 3000 --lr 3e-4 --min_lr_ratio 0.05 --eval_every 200 --out ckpt.npz

## 4. Export the browser weights and preview a sample

In [ ]:
!python export_web.py --ckpt ckpt.npz --out web/model.json
!python sample.py --ckpt ckpt.npz --prompt 'The history of' --tokens 200 --temperature 0.8 --top_p 0.92 --repetition_penalty 1.3

## 5. Download `model.json`

Then send this file back to deploy it to the live app (or drop it into `llm-from-scratch/web/model.json` in the repo and push).

In [ ]:
from google.colab import files
print('model.json size:', os.path.getsize('web/model.json'), 'bytes')
files.download('web/model.json')